# SGGF-Net Training Notebook (Google Colab - T4 GPU)

**3-Stage Training Strategy optimized for Colab T4 GPU**

- Stage 1: Baseline Faster-RCNN (8 epochs, ~15-20 min)
- Stage 2: Enable GFEM (6 epochs, ~12-15 min)
- Stage 3: Enable NDPA + ARPM (4 epochs, ~8-10 min)

**Total time: ~35-45 minutes on T4 GPU**

**Setup:**
1. Runtime → Change runtime type → GPU (T4)
2. Run all cells

In [ ]:
# Step 1: Mount Drive and clone repository
import os
import subprocess
from google.colab import drive

print("=" * 70)
print("SETUP")
print("=" * 70)

# Mount Drive
try:
    if os.path.exists('/content/drive/MyDrive'):
        print('✓ Google Drive already mounted')
    else:
        drive.mount('/content/drive', force_remount=False)
        print('✓ Google Drive mounted')
except Exception as e:
    print(f'⚠ Drive mounting failed: {e}')
    print('  Continuing without Drive (checkpoints will be saved locally)')

# Clone repository
if not os.path.exists('SGGF-Net'):
    print('\n📦 Cloning repository...')
    result = subprocess.run(['git', 'clone', 'https://github.com/HarishSankarK/SGGF-Net.git'], 
                           capture_output=True, text=True)
    if result.returncode != 0:
        print(f'❌ Git clone failed: {result.stderr}')
    else:
        print('✓ Repository cloned')
        os.chdir('SGGF-Net')
else:
    print('✓ Repository already exists, using existing')
    os.chdir('SGGF-Net')

# Verify we're in the right place
if not os.path.exists('scripts/train.py'):
    print(f'❌ ERROR: scripts/train.py not found in {os.getcwd()}')
    print('Please check repository structure')
else:
    print(f'\n✓ Working directory: {os.getcwd()}')
    print(f'✓ Found train.py at: {os.path.abspath("scripts/train.py")}')
    
print("=" * 70)

In [ ]:
# Step 2: Install dependencies and verify GPU
import sys
import subprocess

print("=" * 70)
print("INSTALLING DEPENDENCIES")
print("=" * 70)

# Install PyTorch with CUDA
print("Installing PyTorch...")
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu118'], check=False)

# Install other dependencies
print("Installing other dependencies...")
subprocess.run([sys.executable, '-m', 'pip', 'install', 'numpy', 'pillow', 'opencv-python', 'tqdm', 'matplotlib', 'scipy'], check=False)

# Verify installation
try:
    import torch
    import torchvision
    import numpy
    import PIL
    import cv2
    print(f"\n✓ PyTorch: {torch.__version__}")
    print(f"✓ Torchvision: {torchvision.__version__}")
    
    if torch.cuda.is_available():
        print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
        print(f"✓ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")
        print(f"✓ CUDA Version: {torch.version.cuda}")
    else:
        print("⚠ No GPU detected! Enable GPU in Runtime settings:")
        print("  Runtime → Change runtime type → GPU (T4)")
        print("  Then re-run this cell")
        
    print("\n✓ All dependencies installed successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("Please restart runtime and try again")
    
print("=" * 70)

## Stage 1: Baseline Faster-RCNN

In [ ]:
# Stage 1 Training
import subprocess
import os
import sys

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Verify we're in the right directory
if not os.path.exists('scripts/train.py'):
    print('❌ ERROR: scripts/train.py not found!')
    print(f'Current directory: {os.getcwd()}')
    print('Please make sure Cell 1 cloned the repository correctly.')
    sys.exit(1)

# Verify dataset exists
if not os.path.exists('data/hit-uav'):
    print('⚠ Dataset not found at data/hit-uav')
    print('The dataset needs to be downloaded. Checking if it exists elsewhere...')
    # Check if dataset might be in a different location
    possible_paths = [
        '/content/drive/MyDrive/datasets/hit-uav',
        '/content/hit-uav',
        'hit-uav'
    ]
    found = False
    for path in possible_paths:
        if os.path.exists(path):
            print(f'✓ Found dataset at: {path}')
            # Create symlink or copy
            os.makedirs('data', exist_ok=True)
            if not os.path.exists('data/hit-uav'):
                os.symlink(os.path.abspath(path), 'data/hit-uav')
                print(f'✓ Created symlink: data/hit-uav -> {path}')
                found = True
                break
    
    if not found:
        print('\n❌ Dataset not found. Please download HIT-UAV dataset:')
        print('  1. Go to: https://github.com/Syo9/HIT-UAV')
        print('  2. Download and extract to: data/hit-uav/')
        print('  3. Or upload to Google Drive and update the path above')
        print('\nExpected structure:')
        print('  data/hit-uav/')
        print('    ├── images/')
        print('    └── annotations/')
        sys.exit(1)
else:
    print(f'✓ Dataset found at: {os.path.abspath("data/hit-uav")}')

print("=" * 70)
print("STAGE 1: BASELINE FASTER-RCNN")
print("=" * 70)
print("Starting training... (output will stream below)")
print("⚠ First batch may take 30-60 seconds to compile on GPU\n")

cmd = ['python', '-u', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', checkpoint_dir, '--stage', '1', '--subset_ratio', '0.35']

# Run with real-time output streaming (unbuffered)
# Use Popen to stream output line by line
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Merge stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
    universal_newlines=True
)

# Stream output in real-time
for line in process.stdout:
    print(line, end='', flush=True)

# Wait for completion
process.wait()

if process.returncode != 0:
    print(f"\n❌ Training failed with exit code {process.returncode}")
    sys.exit(1)
else:
    print("\n✓ Stage 1 training completed successfully!")

## Stage 2: Enable GFEM

In [ ]:
# Stage 2 Training
import subprocess
import os
import sys

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
stage1_best = os.path.join(checkpoint_dir, 'stage1_best.pth')

if not os.path.exists(stage1_best):
    print(f"⚠ {stage1_best} not found. Run Stage 1 first!")
    sys.exit(1)

print("=" * 70)
print("STAGE 2: ENABLE GFEM")
print("=" * 70)
print("Starting training... (output will stream below)")
print("⚠ First batch may take 30-60 seconds to compile on GPU\n")

cmd = ['python', '-u', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', checkpoint_dir, '--stage', '2', '--resume', stage1_best, '--subset_ratio', '0.35']

# Run with real-time output streaming (unbuffered)
# Use Popen to stream output line by line
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Merge stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
    universal_newlines=True
)

# Stream output in real-time
for line in process.stdout:
    print(line, end='', flush=True)

# Wait for completion
process.wait()

if process.returncode != 0:
    print(f"\n❌ Training failed with exit code {process.returncode}")
    sys.exit(1)
else:
    print("\n✓ Stage 2 training completed successfully!")

## Stage 3: Enable NDPA + ARPM

In [ ]:
# Stage 3 Training
import subprocess
import os
import sys

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
stage2_best = os.path.join(checkpoint_dir, 'stage2_best.pth')

if not os.path.exists(stage2_best):
    print(f"⚠ {stage2_best} not found. Run Stage 2 first!")
    sys.exit(1)

print("=" * 70)
print("STAGE 3: ENABLE NDPA + ARPM")
print("=" * 70)
print("Starting training... (output will stream below)")
print("⚠ First batch may take 30-60 seconds to compile on GPU\n")

cmd = ['python', '-u', 'scripts/train.py', '--data_dir', 'data/hit-uav', '--num_classes', '6', '--checkpoint_dir', checkpoint_dir, '--stage', '3', '--resume', stage2_best, '--subset_ratio', '0.35']

# Run with real-time output streaming (unbuffered)
# Use Popen to stream output line by line
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Merge stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
    universal_newlines=True
)

# Stream output in real-time
for line in process.stdout:
    print(line, end='', flush=True)

# Wait for completion
process.wait()

if process.returncode != 0:
    print(f"\n❌ Training failed with exit code {process.returncode}")
    sys.exit(1)
else:
    print("\n✓ Stage 3 training completed successfully!")

## Evaluate Final Model

In [ ]:
# Evaluation
import subprocess
import os
import sys
import torch

checkpoint_dir = '/content/drive/MyDrive/SGGF-Net-checkpoints'
final_checkpoint = os.path.join(checkpoint_dir, 'stage3_best.pth')
if not os.path.exists(final_checkpoint):
    final_checkpoint = os.path.join(checkpoint_dir, 'stage3_latest.pth')

if not os.path.exists(final_checkpoint):
    print(f"⚠ {final_checkpoint} not found. Complete all stages first!")
    sys.exit(1)

device_str = 'cuda' if torch.cuda.is_available() else 'cpu'
print("=" * 70)
print("EVALUATION")
print("=" * 70)
print("Starting evaluation... (output will stream below)\n")

cmd = ['python', '-u', 'scripts/evaluate.py', '--dataset', 'hituav', '--data_dir', 'data/hit-uav', '--checkpoint', final_checkpoint, '--num_classes', '6', '--batch_size', '1', '--max_size', '640', '--split', 'test', '--device', device_str]

# Run with real-time output streaming (unbuffered)
# Use Popen to stream output line by line
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # Merge stderr into stdout
    text=True,
    bufsize=1,  # Line buffered
    universal_newlines=True
)

# Stream output in real-time
for line in process.stdout:
    print(line, end='', flush=True)

# Wait for completion
process.wait()

if process.returncode != 0:
    print(f"\n❌ Evaluation failed with exit code {process.returncode}")
    sys.exit(1)
else:
    print("\n✓ Evaluation completed successfully!")